## **Sliding Puzzle Search Assignment**

### **Problem Definition**

You are given an **n×n sliding puzzle** (the *n²−1 puzzle*) containing the numbers **1 to n²−1** and one **empty tile**.
A move consists of sliding one of the adjacent tiles (**up, down, left, or right**) into the empty space.
The goal is to arrange the numbers in **ascending order from left to right and top to bottom**, with the empty tile in the last position.

If the puzzle is **unsolvable**, the program must output **“Unsolvable.”**


In [ ]:
import numpy as np

import heapq, random


class Stack:
    "A container with a last-in-first-out (LIFO) queuing policy."

    def __init__(self):
        self.list = []

    def push(self, item):
        "Push 'item' onto the stack"
        self.list.append(item)

    def pop(self):
        "Pop the most recently pushed item from the stack"
        return self.list.pop()

    def is_empty(self):
        "Returns true if the stack is empty"
        return len(self.list) == 0

    def size(self):
        return len(self.list)


class Queue:
    "A container with a first-in-first-out (FIFO) queuing policy."

    def __init__(self):
        self.list = []

    def push(self, item):
        "Enqueue the 'item' into the queue"
        self.list.insert(0, item)

    def pop(self):
        """
        Dequeue the earliest enqueued item still in the queue. This
        operation removes the item from the queue.
        """
        return self.list.pop()

    def is_empty(self):
        "Returns true if the queue is empty"
        return len(self.list) == 0
    
    def size(self):
        return len(self.list)


class PriorityQueue:
    """
    Implements a priority queue data structure. Each inserted item
    has a priority associated with it and the client is usually interested
    in quick retrieval of the lowest-priority item in the queue. This
    data structure allows O(1) access to the lowest-priority item.
    """

    def __init__(self):
        self.heap = []
        self.count = 0

    def push(self, item, priority):
        entry = (priority, self.count, item)
        heapq.heappush(self.heap, entry)
        self.count += 1

    def pop(self):
        (_, _, item) = heapq.heappop(self.heap)
        return item

    def is_empty(self):
        return len(self.heap) == 0

    def update(self, item, priority):
        # If item already in priority queue with higher priority, update its priority and rebuild the heap.
        # If item already in priority queue with equal or lower priority, do nothing.
        # If item not in priority queue, do the same thing as self.push.
        for index, (p, c, i) in enumerate(self.heap):
            if i == item:
                if p <= priority:
                    break
                del self.heap[index]
                self.heap.append((priority, c, item))
                heapq.heapify(self.heap)
                break
        else:
            self.push(item, priority)

    def size(self):
        return len(self.heap)

In [2]:
goal_state = np.arange(1, 10).reshape((3, 3))
start_state = np.array([1, 2, 3, 4, 5, 6, 7, 9, 8]).reshape(3, 3)
print("goal state:\n", goal_state)
print("start state:\n", start_state)

goal state:
 [[1 2 3]
 [4 5 6]
 [7 8 9]]
start state:
 [[1 2 3]
 [4 5 6]
 [7 9 8]]


In [3]:
nine_row = np.where(start_state == 9)[0][0]
nine_col = np.where(start_state == 9)[1][0]
print(nine_row)
print(nine_col)

2
1


In [5]:
def get_successors(state):
    successors = []
    nine_row = np.where(state == 9)[0][0]
    nine_col = np.where(state == 9)[1][0]
    cost = 1
    if nine_row > 0:
        action = "U"
        next_state = state.copy()
        next_state[nine_row, nine_col] = state[nine_row - 1, nine_col]
        next_state[nine_row - 1, nine_col] = 9
        successors.append((next_state, action, cost))
    if nine_row < 2:
        action = "D"
        next_state = state.copy()
        next_state[nine_row, nine_col] = state[nine_row + 1, nine_col]
        next_state[nine_row + 1, nine_col] = 9
        successors.append((next_state, action, cost))
    if nine_col > 0:
        action = "L"
        next_state = state.copy()
        next_state[nine_row, nine_col] = state[nine_row, nine_col - 1]
        next_state[nine_row, nine_col - 1] = 9
        successors.append((next_state, action, cost))
    if nine_col < 2:
        action = "R"
        next_state = state.copy()
        next_state[nine_row, nine_col] = state[nine_row, nine_col + 1]
        next_state[nine_row, nine_col + 1] = 9
        successors.append((next_state, action, cost))
    return successors

In [6]:
def is_goal_state(state):
    return (state == goal_state).all()

# un-informed search [25 points]

Implement the **Breadth-First Search (BFS)** and **Depth-First Search (DFS)** algorithms to solve the puzzle.
Compare their behavior in terms of:

* Number of explored nodes
* Memory consumption
* Optimality of the found solution

In [7]:
algorithm = "DFS"
# algorithm = "BFS"


if algorithm == "BFS":
    fringe = Queue() # which data structure is needed for BFS?
elif algorithm == "DFS": 
    fringe = Stack() # which one for BFS?

explored = set() # empty set of explored nodes
path = [] # empty list of actions
max_fringe_size = 0
start_state = np.array([1,2,3,4,5,6,7,9,8]).reshape(3,3) # let pick an easy to solve start state
fringe.push(( start_state , path )) # push (start state , path) to fringe

print("start state: \n",start_state)

while not fringe.is_empty(): # while fringe is not empty
    state , path = fringe.pop() # pop a successor from fringe
    max_fringe_size = max(max_fringe_size, fringe.size())
    hashable_state = tuple(state.ravel()) # in python 2d arrays cannot be hashed, therefore cannot be added to set --> cast state to 1d tuple
    if is_goal_state(state):# check if current node is the goal node
        print("victory...")
        break
    if hashable_state not in explored: # if the node poped from fringed is NOT already explored
        explored.add( hashable_state ) # add it to explored set
        for child_state , action , cost in get_successors( state ): # and get its children : child node (child state), child action (action that leads to that node), child cost (cost of that action) 
            new_path = path + [ action ] # new path = current path + new action --> we put [] around new_action to cast new_action to a list element
            # If state not already visited:
            if tuple(child_state.ravel()) not in explored:
                fringe.push( ( child_state , new_path ) ) # tuple of child's state and new path     
            max_fringe_size = max(max_fringe_size, fringe.size())
            
print('path:', path)
print("path length:", len(path))
print("explored length:", len(explored))
print("max fringe size:", max_fringe_size)

start state: 
 [[1 2 3]
 [4 5 6]
 [7 9 8]]
victory...
path: ['R']
path length: 1
explored length: 1
max fringe size: 3


# informed search [25 points]

Implement the **A*** algorithm using a suitable **heuristic function** (e.g., Manhattan distance or number of misplaced tiles).
Evaluate its performance based on the **number of explored states**:

* Fewer than **17,114 states** → full score
* More than **130,460 states** → no score
* Intermediate values are **graded linearly** according to the following formula:


For example, if you explore **100,000 states**, your score is calculated as below \
$ 25 - 25 \times \frac{(100000 - 17114)}{(130460 - 17114)}  $


In [8]:
def heuristic(state):
    nine_row = np.where(state==9)[0][0]
    nine_col = np.where(state==9)[1][0]
    # think about a good heuristic
    cost = 0
    for i in range(3):
        for j in range(3):
            value = state[i, j]
            if value != 9:  # skip the empty tile
                goal_row, goal_col = divmod(value - 1, 3)
                cost += abs(goal_row - i) + abs(goal_col - j)

    return cost

# start_state = np.random.permutation(np.arange(1,10)).reshape((3,3))
start_state = np.array([[8, 5, 4],[6, 1, 7],[9, 3, 2]])
print("start state: \n",start_state)

fringe = PriorityQueue()
explored = set() # empty set of explored nodes
path = [] # empty list of actions
g_state = 0 # what is g(start state) ?
h_state = heuristic(start_state) # what is h(start state)?
f_state = g_state + h_state # f_state = g_state + h_state
max_fringe_size = 0
fringe.push(( start_state , path , g_state ), f_state ) # push ( (state , path ,g_state) , f_state ) to fringe. check the push function in util.priorityQueue. The second input argument (f_state) defines the priority

while not fringe.is_empty(): # while fringe is not empty
    state , path , g_state  = fringe.pop() # pop a successor from fringe
    max_fringe_size = max(max_fringe_size, fringe.size())
    hashable_state = tuple(state.ravel())
    h_state = heuristic(state) # what is ur heuristic
    f_state = g_state + h_state # f_state = g_state + h_state
    if is_goal_state(state):# check if current node is the goal node
        print("victory...")
        break
    if hashable_state not in explored: # if the node poped from fringed is NOT already explored
        explored.add(hashable_state) # add it to explored set
        for child_state , action , cost in get_successors(state) : # get its children -> next state,  next action , cost of action
            new_path = path + [ action ] # new path = current path + new action
            g_next_state = g_state + cost   # g_next_state = cost till now + cost of new action
            h_next_state = heuristic( child_state ) # h_next_state (for USC u can use the nullHeuristic function, for A* u can use other heuristics)
            f_next_state = g_next_state + h_next_state # f_next_state = g_next_state + h_next_state
            fringe.push(( child_state , new_path , g_next_state ), f_next_state ) # check the push function in util.priorityQueue
            max_fringe_size = max(max_fringe_size, fringe.size())

print('path:', path)
print("path length:", len(path))
print("explored length:", len(explored))
print("max fringe size:", max_fringe_size)

start state: 
 [[8 5 4]
 [6 1 7]
 [9 3 2]]
victory...
path: ['U', 'R', 'R', 'U', 'L', 'L', 'D', 'R', 'R', 'D', 'L', 'L', 'U', 'R', 'U', 'R', 'D', 'D', 'L', 'U', 'U', 'R', 'D', 'D']
path length: 24
explored length: 358
max fringe size: 501


In [9]:
# Enjoy watching how u found the solution :)


def get_next_state(state, action):
    nine_row = np.where(state == 9)[0][0]
    nine_col = np.where(state == 9)[1][0]
    if action == "U":
        next_state = state.copy()
        next_state[nine_row, nine_col] = state[nine_row - 1, nine_col]
        next_state[nine_row - 1, nine_col] = 9
    elif action == "D":
        next_state = state.copy()
        next_state[nine_row, nine_col] = state[nine_row + 1, nine_col]
        next_state[nine_row + 1, nine_col] = 9
    if action == "L":
        next_state = state.copy()
        next_state[nine_row, nine_col] = state[nine_row, nine_col - 1]
        next_state[nine_row, nine_col - 1] = 9
    if action == "R":
        next_state = state.copy()
        next_state[nine_row, nine_col] = state[nine_row, nine_col + 1]
        next_state[nine_row, nine_col + 1] = 9
    return next_state


state = start_state.copy()
print(state)
for action in path:
    state = get_next_state(state, action)
    print(action)
    print(state)

[[8 5 4]
 [6 1 7]
 [9 3 2]]
U
[[8 5 4]
 [9 1 7]
 [6 3 2]]
R
[[8 5 4]
 [1 9 7]
 [6 3 2]]
R
[[8 5 4]
 [1 7 9]
 [6 3 2]]
U
[[8 5 9]
 [1 7 4]
 [6 3 2]]
L
[[8 9 5]
 [1 7 4]
 [6 3 2]]
L
[[9 8 5]
 [1 7 4]
 [6 3 2]]
D
[[1 8 5]
 [9 7 4]
 [6 3 2]]
R
[[1 8 5]
 [7 9 4]
 [6 3 2]]
R
[[1 8 5]
 [7 4 9]
 [6 3 2]]
D
[[1 8 5]
 [7 4 2]
 [6 3 9]]
L
[[1 8 5]
 [7 4 2]
 [6 9 3]]
L
[[1 8 5]
 [7 4 2]
 [9 6 3]]
U
[[1 8 5]
 [9 4 2]
 [7 6 3]]
R
[[1 8 5]
 [4 9 2]
 [7 6 3]]
U
[[1 9 5]
 [4 8 2]
 [7 6 3]]
R
[[1 5 9]
 [4 8 2]
 [7 6 3]]
D
[[1 5 2]
 [4 8 9]
 [7 6 3]]
D
[[1 5 2]
 [4 8 3]
 [7 6 9]]
L
[[1 5 2]
 [4 8 3]
 [7 9 6]]
U
[[1 5 2]
 [4 9 3]
 [7 8 6]]
U
[[1 9 2]
 [4 5 3]
 [7 8 6]]
R
[[1 2 9]
 [4 5 3]
 [7 8 6]]
D
[[1 2 3]
 [4 5 9]
 [7 8 6]]
D
[[1 2 3]
 [4 5 6]
 [7 8 9]]


###  Moving the Goal Test [10 points]

Currently, `is_goal(state)` is checked **when a node is popped** from the fringe.
Imagine instead checking it **right after generating successors** (before pushing them onto the fringe).

**Without changing your code**, answer conceptually:

1. Would this change the **solution or order of discovery** for **BFS**, **UCS**, or **A*** (with an admissible heuristic)?

- **BFS**: No change in solution optimality, but could change order of discovery
- **UCS**: Could change solution optimality (potentially non-optimal)
- **A***: Could change solution optimality (potentially non-optimal)

2. **Why or why not?** Relate your answer to how nodes are **generated vs. popped** and how that affects **optimality**.

- **BFS** explores by depth levels, so checking after generation vs. popping won't affect optimality since all nodes at the same depth have equal cost
- **UCS and A*** rely on the priority queue to ensure the lowest-cost path is expanded first. Checking the goal immediately after generation might return a suboptimal solution because:
  - A higher-cost path to the goal might be generated before the optimal path
  - The algorithm might find a goal state through a more expensive path before the optimal path is expanded
  - The priority queue ensures we expand the most promising node first, which guarantees optimality when we check on pop


### Informed vs Uninformed Search [5 points]:

In uninformed search, a state is added to the fringe (or frontier) only if it has not been visited before. However, in the A* algorithm (an informed search), a state may be added to the fringe even if it has been visited previously.

**Why A* may revisit states:**
- In uninformed search, once a state is visited, any other path to that state would be equal or longer
- In A*, different paths to the same state can have different f-costs (g + h)
- If we find a new path with lower g-cost (actual cost) to an already-visited state, we should reconsider it because it might lead to a better overall solution
- This is handled by the `update` method in your PriorityQueue


### Fringe Growth [5 points]

Estimate the **maximum possible size** of the **fringe** during the search.

**Maximum possible fringe size estimate:**
- For an 8-puzzle, the total state space is 9! = 362,880 possible states
- In worst-case BFS, the fringe could contain nearly all states at the maximum depth level
- Practical maximum fringe size is limited by the branching factor (2-4) and search depth
- From your A* results: maximum fringe size was 501, which is reasonable
- Theoretical maximum could approach ~50,000-100,000 states for harder puzzles

### Tree search or graph search? [5 points]

State clearly whether your implementation is **tree search** or **graph search**.

The implementation is **graph search** because:
- You maintain an `explored` set to track visited states
- You check `if tuple(child_state.ravel()) not in explored` before adding to fringe
- This prevents revisiting states and infinite loops
- Tree search would not have the explored set and could revisit states

### consistency and admissiblity [10 points]:

What can be said about the consistency and admissibility of your heuristic? Is A* guaranteed to find the optimal solution when using your heuristic?

**The Manhattan distance heuristic:**
- **Admissible**: Yes - never overestimates the actual cost to reach the goal
- **Consistent**: Yes - satisfies h(n) ≤ c(n, n') + h(n') for all transitions

**Guarantees for A*:**
- With an admissible and consistent heuristic, A* is guaranteed to find the optimal solution
- The implementation will always find the shortest path to the goal
- Consistency ensures that when a node is expanded, we've found the optimal path to that node

### Comparison [15 points]

Assume we have run the search using the following five algorithms and obtained these results:

| Algorithm | Path Length | Nodes Expanded | Max Fringe Size |
| --------- | ----------- | -------------- | --------------- |
| **BFS**   | 24          | 130,461        | 24,053          |
| **DFS**   | 33,890      | 152,779        | 42,826          |
| **UCS**   | 24          | 130,461        | 61,364          |
| **IDDFS** | 24          | 3,327,638      | 25              |
| **A***    | 24          | 359            | 501             |

Now, compare the algorithms in terms of **memory consumption**, **execution time**, **optimality**, and **completeness**

**Memory Consumption** (Max Fringe Size):
- **Best**: IDDFS (25) - only stores current path
- **Good**: A* (501) - focused search
- **Moderate**: BFS (24,053) - stores entire breadth levels
- **Worst**: DFS (42,826) and UCS (61,364) - can store many unpromising nodes

**Execution Time** (Nodes Expanded):
- **Best**: A* (359) - heuristic guides search efficiently
- **Good**: BFS/UCS (130,461) - systematic but uninformed
- **Poor**: DFS (152,779) - can explore deep, unpromising paths
- **Worst**: IDDFS (3,327,638) - repeats work at each depth level

**Optimality:**
- **Optimal**: BFS, UCS, IDDFS, A* (all found path length 24)
- **Non-optimal**: DFS (found path length 33,890)

**Completeness:**
- **All complete** for finite state spaces like the 8-puzzle

**Overall Assessment:**
- **A*** is clearly superior - optimal with minimal node expansion
- **BFS** is reliable but memory-intensive
- **UCS** behaves like BFS for uniform cost problems
- **DFS** is memory-efficient but can find suboptimal solutions
- **IDDFS** guarantees optimality but is computationally expensive